### วิธีใช้งาน
1. รันทุกเซลล์จากเซลล์บนสุดก่อน
2. Copy path ใส่ให้ครบทุกช่อง
3. กดดูตัวอย่างก่อนเปลี่ยนชื่อ
4. ตรวจสอบ แล้วกดยืนยัน

In [1]:

%pip install easyocr
import os # จัดการไฟล์
import re # ดึงตัวเลขตัวอักษรและลบตัวที่ไม่ต้องการตามเงื่อนไข
import cv2 # ปรับภาพ
import pandas as pd # อ่านไฟล์ xlsx
import numpy as np # แปลงโครงสร้างเป็น Array
import easyocr # อ่านภาพ
from PIL import Image # เปิดภาพ
from google.colab import drive # ดึงข้อมูลไดร์ฟเข้าสู่ colab
from google.colab.patches import cv2_imshow # แสดงผลออกมาเป็นรูปภาพ
from IPython.display import display, HTML #สร้าง gui

# Mount Google Drive
drive.mount('/content/drive')

# โหลด EasyOCR / กำหนดค่า
reader = easyocr.Reader(['en'])
CROP_BOX = (453, 908, 514, 930)
SHOW_DEBUG_IMAGE = False

# ==========================================
# 1. ฟังก์ชัน Pre-processing
# ==========================================
#ฟังก์ชันแปลงภาพเป็นขาวดำ
def pre_process_low_contrast_image(img_np_uint8):
    if img_np_uint8.ndim == 2:
        gray = img_np_uint8
    elif img_np_uint8.ndim == 3:
        if img_np_uint8.shape[2] == 4:
            gray = cv2.cvtColor(img_np_uint8, cv2.COLOR_RGBA2GRAY)
        elif img_np_uint8.shape[2] == 3:
            gray = cv2.cvtColor(img_np_uint8, cv2.COLOR_RGB2GRAY)
        else:
            gray = img_np_uint8
    else:
        gray = img_np_uint8

    resized_gray = cv2.resize(gray, None, fx=3, fy=3, interpolation=cv2.INTER_CUBIC)
    _, binarized = cv2.threshold(resized_gray, 200, 255, cv2.THRESH_BINARY)
    kernel = np.ones((2, 2), np.uint8)
    eroded = cv2.erode(binarized, kernel, iterations=1)

    if SHOW_DEBUG_IMAGE:
        cv2_imshow(eroded)

    return eroded

#ฟังก์ชันอ่านค่ากำลังขยายจากภาพ
def extract_magnification_from_image(image_path, crop_box):
    try:
        image = Image.open(image_path)
        cropped_img = image.crop(crop_box) if crop_box else image
        img_np = np.array(cropped_img).astype(np.uint8)
        processed_image = pre_process_low_contrast_image(img_np)

        results = reader.readtext(
            processed_image,
            allowlist='0123456789xX',
            text_threshold=0.3,
            detail=0
        )
        raw_text = "".join(results).lower().replace('x', '')

        match = re.search(r'(\d+(?:\.\d+)?)', raw_text)
        if match:
            return float(match.group(1))
    except Exception as e:
        print(f"[WARNING] เกิดข้อผิดพลาดในการอ่าน OCR จากไฟล์ {os.path.basename(image_path)}: {e}")

    return 0.0

#ฟังก์ชันเปลี่ยนอักษรต้องห้ามเป็นให้เป็น _ (underscore)
def sanitize_filename(name):
    if pd.isna(name):
        return ""
    name_str = str(name).strip()
    return re.sub(r'[\\/*?:"<>|]', '_', name_str)

#ฟังก์ชันปรับแต่งเลข ID
def clean_id(val):
    if pd.isna(val):
        return ""
    val_str = str(val).strip()
    if val_str.endswith('.0'):
        val_str = val_str[:-2]
    return val_str

#ฟังก์ชันแปลง format วันที่ ให้เป็น YYYY-MM-DD
def normalize_date(val):
    if pd.isna(val) or val is None or str(val).strip() == '':
        return ""

    if isinstance(val, (pd.Timestamp, np.datetime64)):
        dt = pd.to_datetime(val)
        y = dt.year - 543 if dt.year > 2500 else dt.year
        return f"{y:04d}-{dt.month:02d}-{dt.day:02d}"

    val_str = str(val).strip()

    if len(val_str) == 6 and val_str.isdigit():
        day = int(val_str[:2])
        month = int(val_str[2:4])
        yy = int(val_str[4:])
        year_be = 2500 + yy if yy > 50 else 2543 + yy
        year_ce = year_be - 543
        return f"{year_ce:04d}-{month:02d}-{day:02d}"

    try:
        val_str_clean = val_str.split(' ')[0]
        parts = re.split(r'[/\-.]', val_str_clean)
        if len(parts) == 3:
            p1, p2, p3 = int(parts[0]), int(parts[1]), int(parts[2])
            if p3 > 2000:
                year = p3 - 543 if p3 > 2500 else p3
                month, day = (p2, p1) if p1 > 12 else (p1, p2)
                return f"{year:04d}-{month:02d}-{day:02d}"
            elif p1 > 2000:
                year = p1 - 543 if p1 > 2500 else p1
                month, day = p2, p3
                return f"{year:04d}-{month:02d}-{day:02d}"
    except Exception:
        pass

    try:
        dt = pd.to_datetime(val_str, errors='coerce')
        if not pd.isna(dt):
            y = dt.year - 543 if dt.year > 2500 else dt.year
            return f"{y:04d}-{dt.month:02d}-{dt.day:02d}"
    except Exception:
        pass

    return val_str

#ฟังก์ชันจัด format กำลังขยาย
def format_mag_str(mag):
    if mag <= 0:
        return "0"
    if mag >= 1000:
        val = mag / 1000.0
        if val.is_integer():
            return f"X{int(val)}k"
        else:
            return f"X{val:.2f}".rstrip('0').rstrip('.') + "k"
    else:
        if mag.is_integer():
            return f"X{int(mag)}"
        else:
            return f"X{mag:.2f}".rstrip('0').rstrip('.')

# ==========================================
# ฟังก์ชันการทำงานหลัก
# ==========================================
def process_renamer_pipeline(excel_path, base_folder_path, dry_run=True):
    if dry_run:
        print("[MODE: DRY RUN] แสดงตัวอย่างผลลัพธ์ (ไม่มีการเปลี่ยนชื่อไฟล์)\n" + "="*80)
    else:
        print("[MODE: EXECUTE] กำลังดำเนินการเปลี่ยนชื่อไฟล์บน Google Drive...\n" + "="*80)

    # 1. เช็ก Path
    if not os.path.exists(excel_path):
        print(f"[WARNING] ไม่พบไฟล์ Excel ที่พาธ: {excel_path}")
        return
    if not os.path.exists(base_folder_path):
        print(f"[WARNING] ไม่พบโฟลเดอร์รูปภาพที่พาธ: {base_folder_path}")
        return

    # 2. อ่าน Excel
    excel_mapping = {}
    try:
        df = pd.read_excel(excel_path, engine='openpyxl')
    except Exception as e:
        df = pd.read_csv(excel_path)

    df.iloc[:, 0] = df.iloc[:, 0].ffill()

    for _, row in df.iterrows():
        raw_date = row.iloc[0]
        norm_date = normalize_date(raw_date)
        key_id = clean_id(row.iloc[1])
        rock_name = sanitize_filename(row.iloc[2])

        if key_id and rock_name and norm_date:
            excel_mapping[(norm_date, key_id)] = rock_name

    print(f"โหลดข้อมูลจาก Excel สำเร็จทั้งหมด {len(excel_mapping)} รายการ")
    print("ตัวอย่าง Key ที่สร้างจาก Excel (5 รายการแรก):")
    for k, v in list(excel_mapping.items())[:5]:
      print(f"   วันที่: '{k[0]}'\t\tID: '{k[1]}'\tชื่อตัวอย่าง: '{v}'")
    print("="*80)

    # 3. วนลูปอ่านโฟลเดอร์
    date_folders = [f for f in os.listdir(base_folder_path) if os.path.isdir(os.path.join(base_folder_path, f))]
    date_folders.sort(key=lambda x: (x[4:], x[2:4], x[:2]))
    print(f"โหลดโฟลเดอร์วันที่สำเร็จทั้งหมด {len(date_folders)} รายการ :")
    for i in list(date_folders):
        print(f"[{i}]\t", end='')

    for folder_name in date_folders:
        current_folder_path = os.path.join(base_folder_path, folder_name)
        all_files = os.listdir(current_folder_path)
        tif_files = sorted([f for f in all_files if f.lower().endswith(('.tif', '.tiff', '.png', '.jpg', '.jpeg'))])
        folder_norm_date = normalize_date(folder_name)

        print("\n" + "="*80)
        print(f"โฟลเดอร์: [{folder_name}]\t\tวันที่: [{folder_norm_date}]\t\t(พบ {len(tif_files)} ไฟล์)")
        print("="*80)

        if not tif_files:
            print("[WARNING] ไม่พบไฟล์รูปภาพในโฟลเดอร์นี้")
            continue

        file_data = []
        for filename in tif_files:
            temp_filename = filename[len(folder_name):] if filename.startswith(folder_name) else filename
            match = re.search(r'(\d+)', temp_filename) or re.search(r'(\d+)', filename)

            if match:
                rock_id = clean_id(match.group(1))
                filepath = os.path.join(current_folder_path, filename)
                print(f"กำลังอ่าน OCR ไฟล์: {filename} ...", end="\r")
                mag = extract_magnification_from_image(filepath, CROP_BOX)

                file_data.append({
                    'original_name': filename,
                    'rock_id': rock_id,
                    'mag': mag,
                    'filepath': filepath
                })

        print(" " * 80, end="\r")

        if not file_data:
            print("[WARNING] ไม่พบไฟล์ที่เข้าเงื่อนไข")
            continue

        current_rock_id = None
        point_num, zoom_num = 1, 1
        used_rock_names = {}

        for i in range(len(file_data)):
            item = file_data[i]
            rock_id = item['rock_id']

            rock_name = excel_mapping.get((folder_norm_date, rock_id))
            if not rock_name:
                print(f"[WARNING] ไม่พบข้อมูลหินของ วันที่ '{folder_name}' ({folder_norm_date}) ID '{rock_id}' ใน Excel")
                rock_name = f"Unknown_{rock_id}"
                continue

            if rock_id != current_rock_id:
                current_rock_id = rock_id
                zoom_num = 1
                point_num = used_rock_names[rock_name] + 1 if rock_name in used_rock_names else 1

            used_rock_names[rock_name] = max(used_rock_names.get(rock_name, 0), point_num)
            ext = os.path.splitext(item['original_name'])[1]
            mag_str = format_mag_str(item['mag'])
            new_filename = f"{rock_name}-{point_num}.{zoom_num}_{mag_str}{ext}"

            html_row = f"""
            <div style="font-family: 'Kanit', sans-serif; font-size: 13px; line-height: 1.8;">
              <table style="width: 100%; border-collapse: collapse; color: #f8fafc;">
                <tr>
                  <td style="width: 10%; color: #475569;">{item['original_name']}</td>
                  <td style="width: 10%; text-align: center; color: #475569;">-></td>
                  <td style="width: 15%; color: #8B1E2D; font-weight: 500;">{new_filename}</td>
                  <td style="width: 5%; text-align: center; color: #475569;">|</td>
                  <td style="width: 20%; color: #475569;">กำลังขยาย: {item['mag']}x</td>
                  <td style="width: 40%; color: #475569;"></td>
                </tr>
              </table>
            </div>
            """
            display(HTML(html_row))

            if not dry_run:
                old_path = item['filepath']
                new_path = os.path.join(current_folder_path, new_filename)
                os.rename(old_path, new_path)

            if i + 1 < len(file_data) and file_data[i+1]['rock_id'] == current_rock_id:
                if item['mag'] <= file_data[i+1]['mag']:
                    zoom_num += 1
                else:
                    point_num += 1
                    zoom_num = 1
                    used_rock_names[rock_name] = max(used_rock_names.get(rock_name, 0), point_num)

    print("\n" + "="*80)
    if dry_run:
        print("แสดงตัวอย่างเสร็จสิ้น โปรดตรวจสอบความถูกต้องอีกครั้ง หากถูกต้องให้กดปุ่ม 'ยืนยันการเปลี่ยนชื่อไฟล์'")
    else:
        print("เปลี่ยนชื่อไฟล์ใน Google Drive สำเร็จ")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 978.2/978.2 kB 29.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.2/296.2 kB 20.7 MB/s eta 0:00:00


Mounted at /content/drive
Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

In [2]:

import warnings
import ipywidgets as widgets
from IPython.display import display, clear_output

# ห้ามแสดงผล warning pin_memory
warnings.filterwarnings("ignore", message=".*pin_memory.*")

# ==========================================
# 1. CSS สำหรับตกแต่ง UI
# ==========================================
css_style = widgets.HTML("""
<style>
    /* import Kanit from Google Font */
    @import url('https://fonts.googleapis.com/css2?family=Kanit:wght@300;400;600&display=swap');

    /* ใช้ฟอนต์ Kanit กับ output ทั้งหมด*/
    * {
        font-family: 'Kanit', sans-serif !important;
    }

    /* ช่องรับข้อความ Input */
    .widget-text input {
        border-radius: 8px !important;
        border: 1px solid #cbd5e1 !important;
        padding: 8px 12px !important;
        background-color: #f8fafc !important;
        outline: none !important;
        transition: all 0.2s ease !important;
    }

    .widget-text input:focus {
        border-color: #2563eb !important;
        background-color: #ffffff !important;
        box-shadow: 0 0 0 3px #2563eb26 !important;
    }

    /* ปุ่มดูตัวอย่าง */
    .btn-preview {
        background: #BFC9D1 !important;;
        color: white !important;
        font-weight: 600 !important;
        border: none !important;
        border-radius: 8px !important;
        box-shadow: 0 4px 6px -1px #BFC9D140 !important;
        transition: all 0.2s ease !important;
    }
    .btn-preview:hover {
        transform: translateY(-1px) !important;
        box-shadow: 0 6px 12px -1px #BFC9D166 !important;
    }

    /* ปุ่มยืนยัน */
    .btn-execute {
        background: #ef4444 !important;
        color: white !important;
        font-weight: 600 !important;
        border: none !important;
        border-radius: 8px !important;
        box-shadow: 0 4px 6px -1px #ef44444d !important;
        transition: all 0.2s ease !important;
    }
    .btn-execute:hover {
        transform: translateY(-1px) !important;
        box-shadow: 0 6px 12px -1px #ef444466 !important;
    }
</style>
""")

# ==========================================
# 2. ส่วนประกอบ Header แบนเนอร์ด้านบน
# ==========================================
header_html = widgets.HTML("""
<div style="
    background: #1e293b;
    color: white;
    padding: 20px 24px;
    border-radius: 12px 12px 0 0;
    box-shadow: 0 4px 6px -1px #0000001a !important;
">
    <h2 style="margin: 0; font-size: 20px; font-weight: 600; color: #f8fafc;">โปรแกรมจัดเรียงและเปลี่ยนชื่อไฟล์</h2>
    <p style="margin: 4px 0 0 0; font-size: 13px; color: #94a3b8;">ระบุไฟล์ Excel และโฟลเดอร์รูปภาพเพื่อเปลี่ยนชื่อ</p>
</div>
""")

# ==========================================
# 3. สร้าง Widgets ช่องรับข้อมูลและปุ่ม
# ==========================================
excel_path_input = widgets.Text(
    value='/content/drive/MyDrive/Script Programming Project/Filename_Template.xlsx',
    placeholder='เช่น /content/drive/MyDrive/.../Filename_Template.xlsx',
    description='Excel Path:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='100%', margin='0 0 12px 0')
)

folder_path_input = widgets.Text(
    value='/content/drive/MyDrive/Script Project',
    placeholder='เช่น /content/drive/MyDrive/...',
    description='Folder Path:',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='100%', margin='0 0 12px 0')
)

btn_preview = widgets.Button(
    description='ดูตัวอย่างก่อนเปลี่ยนชื่อ',
    icon='eye',
    layout=widgets.Layout(width='49%', height='42px')
)
btn_preview.add_class('btn-preview')  # ดึง CSS สีส้มมาใส่

btn_execute = widgets.Button(
    description='ยืนยันการเปลี่ยนชื่อไฟล์',
    icon='check',
    layout=widgets.Layout(width='49%', height='42px')
)
btn_execute.add_class('btn-execute')  # ดึง CSS สีแดงมาใส่

# กล่อง Terminal แสดง Output
output_area = widgets.Output(
    layout=widgets.Layout(
        border='1px solid #1e293b',
        padding='12px',
        margin='15px 0 0 0',
        min_height='280px',
        height='auto',
        background_color='#0f172a',  # พื้นหลังสีเข้ม
        overflow_y='auto',
        border_radius='8px'
    )
)

# ==========================================
# 4. Event Handlers
# ==========================================
def on_preview_clicked(b):
    with output_area:
        clear_output()
        process_renamer_pipeline(excel_path_input.value.strip(), folder_path_input.value.strip(), dry_run=True)

def on_execute_clicked(b):
    with output_area:
        clear_output()
        process_renamer_pipeline(excel_path_input.value.strip(), folder_path_input.value.strip(), dry_run=False)

btn_preview.on_click(on_preview_clicked)
btn_execute.on_click(on_execute_clicked)

# ==========================================
# 5. จัด Layout การ์ดและการแสดงผล
# ==========================================
button_box = widgets.HBox(
    [btn_preview, btn_execute],
    layout=widgets.Layout(justify_content='space-between', margin='8px 0px')
)

form_card = widgets.VBox(
    [excel_path_input, folder_path_input, button_box, output_area],
    layout=widgets.Layout(
        background_color='#ffffff',
        padding='20px'
    )
)

main_app = widgets.VBox(
    [css_style, header_html, form_card],
    layout=widgets.Layout(
        width='100%',
        margin='0',
        border_radius='16px',
        overflow='hidden',
        border='1px solid #e2e8f0',
        box_shadow='0 10px 15px -3px #0000000d'
    )
)

display(main_app)